# E3: piloto GPU de desarrollo común

Selecciona GPU en Entorno de ejecución y habilita GITHUB_TOKEN en Secretos de Colab (Contents: Read and write). Nunca pegues el token en código o chat. Ejecuta la celda.

Compara el candidato V3 y sus controles con tres semillas en 532 ventanas de ajuste y 116 de validación; no abre los miembros de prueba/calibración ni evalúa conjuntos adicionales. Conserva los entrenamientos históricos. Esta nueva propuesta sí requiere un piloto propio.

Reutiliza el runtime aislado Python 3.12.12/Torch 2.8.0+cu126 del preflight; la instalación inicial puede tardar varios minutos y ocupar varios GB. No cambia el kernel. Cada tres trabajos guarda checkpoints y resultados en ramas nuevas de GitHub. Reejecutar conserva la ruta y reanuda; no borres o alteres los checkpoints. Guarda las URLs de cada parte y del índice final.

Es desarrollo, no confirmación. Tres semillas/controles no dimensionan la batería de potencia. No ampliar automáticamente tras ver resultados: revisar primero la comparación y la bitácora, luego fijar la familia y el protocolo. R1/R3 no acreditadas. Asistencia de IA declarada.


In [ ]:
import json, urllib.request
from datetime import datetime, timezone
from pathlib import Path

import os
from getpass import getpass

context = globals().get("_E3_CLOSURE_GPU_CONTEXT")
if context:
    source = context["source"]
else:
    with urllib.request.urlopen("https://api.github.com/repos/jassan-bit/DeepLearning_202630/git/ref/heads/main", timeout=30) as response:
        source = json.load(response)["object"]["sha"]
if len(source) != 40 or any(c not in "0123456789abcdef" for c in source):
    raise ValueError("No se obtuvo un SHA completo válido")
print("Código fijado:", source)
if not context:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    context = {"source":source, "output":"/content/e3-closure-pilot-" + run_id}
    _E3_CLOSURE_GPU_CONTEXT = context
url = "https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/" + source + "/experiments/e3_cierre_integral_20261118/campaign/colab.py"
with urllib.request.urlopen(url, timeout=30) as response:
    code = response.read().decode("utf-8")
launcher = {"__name__":"e3_common_gpu_launcher_functions"}
exec(compile(code, url, "exec"), launcher)
token = (os.environ.get("GITHUB_TOKEN") or globals().get("_E3_GITHUB_TOKEN") or "").strip()
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        token = ""
if not token:
    token = getpass("GITHUB_TOKEN (Contents: Read and write para este repositorio): ").strip()
_E3_GITHUB_TOKEN = token
result = launcher["run"](source, token, suite="controls", seeds=(20261118,20261119,20261120), resume_output=context["output"])
_E3_CLOSURE_GPU_RESULT = result
print(json.dumps(result, ensure_ascii=False, indent=2))
